# Model Training Workflow (Refatorado)

Este notebook segue o padrao do projeto: documentacao no inicio e logica de negocio isolada em `scripts/`.

## Objetivo
- Treinar e avaliar um modelo `lgbm_model` para predicao de evasao.
- Reproduzir o fluxo usado como referencia no experimento original.

## Fluxo
1. Carregar dataset processado.
2. Validar nulos e disponibilidade de features.
3. Preparar split estratificado.
4. Rodar validacao cruzada.
5. Treinar modelo final e avaliar no teste.

## Observacao
As funcoes usadas aqui estao em `scripts/model_training_analysis.py` e sao cobertas por testes em `tests/scripts/test_model_training_analysis.py`.


In [ ]:
from pathlib import Path

import pandas as pd

from scripts.data_processing import analise_nulos
from scripts.model_training_analysis import (
    criar_modelo_lgbm,
    executar_validacao_cruzada_lgbm,
    preparar_dados_modelagem,
    resumir_resultados_cv,
    treinar_e_avaliar_lgbm,
)
from src.feature_engineering import MODEL_FEATURE_COLUMNS


In [ ]:
data_path = Path("../data/processed/df_model_concatenado.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Arquivo nao encontrado: {data_path}")

df_loaded = pd.read_csv(data_path)
display(df_loaded.head())
display(analise_nulos(df_loaded).head(10))


In [ ]:
features_disponiveis = [c for c in MODEL_FEATURE_COLUMNS if c in df_loaded.columns]
if len(features_disponiveis) < 2:
    features_disponiveis = None

X_train, X_test, y_train, y_test, features_finais = preparar_dados_modelagem(
    df=df_loaded,
    target_coluna="evadiu",
    feature_columns=features_disponiveis,
    test_size=0.2,
    random_state=42,
)

print(f"Features usadas: {len(features_finais)}")
print(features_finais)


In [ ]:
lgbm_params = {
    "n_estimators": 100,
    "learning_rate": 0.01,
    "max_depth": 4,
    "num_leaves": 31,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
}

modelo = criar_modelo_lgbm(parametros=lgbm_params, random_state=42)
resultados_cv = executar_validacao_cruzada_lgbm(
    X_train=X_train,
    y_train=y_train,
    modelo=modelo,
    n_splits=5,
    random_state=42,
)

display(resumir_resultados_cv(resultados_cv))


In [ ]:
resultado_final = treinar_e_avaliar_lgbm(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    modelo=modelo,
)

print("Metricas no teste:")
for nome, valor in resultado_final["test_metrics"].items():
    print(f"- {nome}: {valor:.4f}")
